### NLP Assignment -8B

In [ ]:
# 2. Design a spell correction model. spell-errors.txt file consists of a correct word and the
# possible errors such as raining: rainning, raning. You are provided with the
# count
# _
# 1edit.txt file that consists of possible errors in 1 edit distance. Similarly use the
# counts or frequencies of the words in count
# _
# unigram.txt file. Use a noisy channel model
# for finding the correct word from an incorrect word. Find the accuracy of your spell
# checker. [You will find all filaes in the Spell-Correction folder.]

unigrams = {}
from pathlib import Path
file = Path("Lab-Assignment-8/Spell-Correction/count_unigram.txt")
with file.open() as f:
    for line in f:
        word, count = line.strip().split()
        unigrams[word] = int(count)

unigrams_total = sum(unigrams.values())



In [2]:
spell_errors = {}
file = Path("Lab-Assignment-8/Spell-Correction/spell-errors.txt")
with file.open() as f:
    for line in f:
        correct_word, errors = line.strip().split(":")
        spell_errors[correct_word] = errors.split(", ")

spell_errors["raining"]


[' rainning', 'raning']

In [3]:
count_1edit = {}

file = Path("Lab-Assignment-8/Spell-Correction/count_1edit.txt")

with file.open(encoding="latin-1") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if not line.strip():
            continue

        edit, count = line.rsplit(maxsplit=1)
        source, target = edit.split("|", maxsplit=1)

        count_1edit[(source, target)] = int(count)

In [4]:
count_1edit[("e", "a")]

749

In [5]:

import string

letters = string.ascii_lowercase


def edits_one(word):
    splits = [(word[:i], word[i:]) for i in range(len(word))]

    deletions = [
        left + right[1:]
        for left, right in splits if right
    ]

    transpositions = [
        left + right[1] + right[0] + right[2:]
        for left, right in splits if len(right) > 1
    ]

    replacements = [
        left + char + right[1:]
        for left, right in splits if right
        for char in letters
    ]

    insertions = [
        left + char + right
        for left, right in splits
        for char in letters
    ]

    return set(
        deletions + transpositions + replacements + insertions
    )


def candidates(word):
    return {
        candidate
        for candidate in edits_one(word)
        if candidate in unigrams
    }


In [6]:
candidates("rainning")

{'raining', 'rainning', 'trainning'}

In [7]:

from collections import defaultdict

substitution_counts = {}
insertion_counts = {}
deletion_counts = {}
transposition_counts = {}
unclassified_counts = {}

file = Path("Lab-Assignment-8/Spell-Correction/count_1edit.txt")

with file.open(encoding="latin-1") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if not line:
            continue

        # The file uses tabs to separate the edit pattern and count.
        # This preserves spaces that are part of an edit pattern.
        edit, count = line.rsplit("\t", maxsplit=1)
        correct, incorrect = edit.split("|", maxsplit=1)
        count = int(count)

        if len(incorrect) == len(correct) + 1:
            insertion_counts[(correct, incorrect)] = count

        elif len(correct) == len(incorrect) + 1:
            deletion_counts[(correct, incorrect)] = count

        elif (
            len(correct) == 2
            and len(incorrect) == 2
            and correct[0] != correct[1]
            and incorrect == correct[::-1]
        ):
            transposition_counts[(correct, incorrect)] = count

        elif len(correct) == len(incorrect) == 1:
            substitution_counts[(correct, incorrect)] = count

        else:
            unclassified_counts[(correct, incorrect)] = count


print("Substitutions:", len(substitution_counts))
print("Insertions:", len(insertion_counts))
print("Deletions:", len(deletion_counts))
print("Transpositions:", len(transposition_counts))
print("Unclassified:", len(unclassified_counts))


Substitutions: 508
Insertions: 385
Deletions: 451
Transpositions: 242
Unclassified: 1


In [8]:
print("Substitution:", substitution_counts.get(("e", "i")))
print("Insertion:", insertion_counts.get(("t", "te")))
print("Deletion:", deletion_counts.get(("te", "t")))
print("Transposition:", transposition_counts.get(("ei", "ie")))

Substitution: 917
Insertion: 478
Deletion: 271
Transposition: 122


In [9]:

def conditional_edit_probabilities(edit_counts):
    """
    Estimate P(incorrect pattern | correct pattern, operation type).
    """
    totals = defaultdict(int)

    for (correct, incorrect), count in edit_counts.items():
        totals[correct] += count

    probabilities = {}

    for (correct, incorrect), count in edit_counts.items():
        probabilities[(correct, incorrect)] = (
            count / totals[correct]
        )

    return probabilities


substitution_probs = conditional_edit_probabilities(
    substitution_counts
)

insertion_probs = conditional_edit_probabilities(
    insertion_counts
)

deletion_probs = conditional_edit_probabilities(
    deletion_counts
)

transposition_probs = conditional_edit_probabilities(
    transposition_counts
)


print("P(i | e, substitution):",
      substitution_probs.get(("e", "i")))

print("P(te | t, insertion):",
      insertion_probs.get(("t", "te")))

print("P(t | te, deletion):",
      deletion_probs.get(("te", "t")))

print("P(ie | ei, transposition):",
      transposition_probs.get(("ei", "ie")))


P(i | e, substitution): 0.3742857142857143
P(te | t, insertion): 0.4438254410399257
P(t | te, deletion): 1.0
P(ie | ei, transposition): 1.0


In [10]:
import string

def edits1(word):
    letters = string.ascii_lowercase

    # Every possible split of the word
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

    # 1. Deletion: remove one character
    deletions = [
        left + right[1:]
        for left, right in splits
        if right
    ]

    # 2. Insertion: insert one character
    insertions = [
        left + char + right
        for left, right in splits
        for char in letters
    ]

    # 3. Substitution: replace one character
    substitutions = [
        left + char + right[1:]
        for left, right in splits
        if right
        for char in letters
    ]

    # 4. Transposition: swap adjacent characters
    transpositions = [
        left + right[1] + right[0] + right[2:]
        for left, right in splits
        if len(right) > 1
    ]

    return set(
        deletions + insertions + substitutions + transpositions
    )

In [11]:
def generate_candidates(word):
    candidates = edits1(word)
    valid_candidates = candidates.intersection(unigrams.keys())

    return sorted(
        valid_candidates,
        key=lambda candidate: unigrams[candidate],
        reverse=True
    )

In [12]:
word = "speling"

candidates = generate_candidates(word)

print("Number of candidates:", len(candidates))
print("Candidates:", candidates[:20])

Number of candidates: 5
Candidates: ['spelling', 'spewing', 'sperling', 'speling', 'seling']


In [ ]:
def word_probability(word):
    return unigrams.get(word, 0) / unigrams_total

In [14]:
def identify_edit(correct, observed):
    # Substitution
    if len(correct) == len(observed):
        differences = [
            i for i in range(len(correct))
            if correct[i] != observed[i]
        ]

        if len(differences) == 1:
            i = differences[0]
            return "substitution", correct[i], observed[i]

        # Transposition of adjacent characters
        if len(differences) == 2:
            i, j = differences

            if (
                j == i + 1
                and correct[i] == observed[j]
                and correct[j] == observed[i]
            ):
                return (
                    "transposition",
                    correct[i:i + 2],
                    observed[i:i + 2]
                )

    # Insertion: observed has one extra character
    if len(observed) == len(correct) + 1:
        for i in range(len(observed)):
            if correct == observed[:i] + observed[i + 1:]:
                return "insertion", correct, observed

    # Deletion: observed is missing one character
    if len(correct) == len(observed) + 1:
        for i in range(len(correct)):
            if observed == correct[:i] + correct[i + 1:]:
                return "deletion", correct, observed

    return None

In [15]:
print(identify_edit("spelling", "speling"))
print(identify_edit("spewing", "speling"))
print(identify_edit("ei", "ie"))

('deletion', 'spelling', 'speling')
('substitution', 'w', 'l')
('transposition', 'ei', 'ie')


In [16]:
def conditional_edit_probabilities(edit_counts):
    from collections import defaultdict

    totals = defaultdict(int)

    for (correct, observed), count in edit_counts.items():
        totals[correct] += count

    probabilities = {}

    for (correct, observed), count in edit_counts.items():
        probabilities[(correct, observed)] = count / totals[correct]

    return probabilities

In [17]:
substitution_probs = conditional_edit_probabilities(substitution_counts)
insertion_probs = conditional_edit_probabilities(insertion_counts)
deletion_probs = conditional_edit_probabilities(deletion_counts)
transposition_probs = conditional_edit_probabilities(transposition_counts)

In [18]:
print("Substitution:", substitution_probs.get(("e", "i")))
print("Insertion:", insertion_probs.get(("t", "te")))
print("Deletion:", deletion_probs.get(("te", "t")))
print("Transposition:", transposition_probs.get(("ei", "ie")))

Substitution: 0.3742857142857143
Insertion: 0.4438254410399257
Deletion: 1.0
Transposition: 1.0


In [19]:
with file.open(encoding="latin-1") as f:
    for line in f:
        pattern, count = line.rstrip("\r\n").rsplit("\t", 1)

        if any(x in pattern for x in ["l|w", "w|l", "ll|l", "l|ll"]):
            print(pattern, count)

l|w 11
w|l 2
wl|w 2
lw|l 1


In [21]:
import math
def get_edit_probability(correct, observed):
    edit = identify_edit(correct, observed)

    if edit is None:
        return 1e-10

    operation, source, target = edit
    key = (source, target)

    probability_tables = {
        "substitution": substitution_probs,
        "insertion": insertion_probs,
        "deletion": deletion_probs,
        "transposition": transposition_probs,
    }

    table = probability_tables[operation]

    # Use a small fallback for edits not found in the table.
    return table.get(key, 1e-10)


def noisy_channel_score(correct, observed):
    """
    Score a candidate correct word given the observed misspelling.
    Uses log probabilities to avoid numerical underflow.
    """
    prior = word_probability(correct)
    likelihood = get_edit_probability(correct, observed)

    if prior <= 0 or likelihood <= 0:
        return -float("inf")

    return math.log(prior) + math.log(likelihood)

In [22]:
def correct_word(word):
    possible_candidates = generate_candidates(word)

    # Include the original word if it is in the vocabulary.
    if word in unigrams:
        possible_candidates.append(word)

    # Remove duplicates
    possible_candidates = set(possible_candidates)

    if not possible_candidates:
        return word, []

    scored_candidates = []

    for candidate in possible_candidates:
        # A word that is already correct receives no edit penalty.
        if candidate == word:
            score = math.log(word_probability(candidate))
        else:
            score = noisy_channel_score(candidate, word)

        scored_candidates.append((candidate, score))

    scored_candidates.sort(key=lambda item: item[1], reverse=True)

    return scored_candidates[0][0], scored_candidates

In [ ]:
word = "rainnng"

prediction, scores = correct_word(word)

print("Input:", word)
print("Predicted correction:", prediction)
print("\nTop candidates:")

for candidate, score in scores[:10]:
    print(f"{candidate:20} {score:.6f}")

Input: raining
Predicted correction: raining

Top candidates:
raining              -12.953741
raising              -14.971016
ruining              -15.507621
gaining              -15.735998
reining              -15.871234
railing              -16.490375
raiding              -17.329816
taining              -18.503483
haining              -19.009226
paining              -20.404778


In [27]:
total = 0
correct_predictions = 0
skipped = 0
wrong_predictions = []

for expected_word, errors in spell_errors.items():
    for misspelled_word in errors:
        prediction, scores = correct_word(misspelled_word)

        if not scores:
            skipped += 1
            continue

        total += 1

        if prediction == expected_word:
            correct_predictions += 1
        else:
            wrong_predictions.append(
                (misspelled_word, expected_word, prediction)
            )

accuracy = (
    correct_predictions / total
    if total > 0
    else 0.0
)

print("Total evaluated errors:", total)
print("Correct predictions:", correct_predictions)
print("Incorrect predictions:", len(wrong_predictions))
print("Skipped errors:", skipped)
print(f"Accuracy: {accuracy:.2%}")

Total evaluated errors: 22262
Correct predictions: 4783
Incorrect predictions: 17479
Skipped errors: 17447
Accuracy: 21.49%


In [37]:
from pathlib import Path

count_1edit = {}

file = Path("Lab-Assignment-8/Spell-Correction/count_1edit.txt")

with file.open(encoding="latin-1") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if not line:
            continue

        edit, count = line.rsplit("\t", maxsplit=1)
        correct, incorrect = edit.split("|", maxsplit=1)

        count_1edit[(correct, incorrect)] = int(count)

print("Total edit patterns:", len(count_1edit))
print("e|i:", count_1edit.get(("e", "i")))
print("t|te:", count_1edit.get(("t", "te")))
print("te|t:", count_1edit.get(("te", "t")))

Total edit patterns: 1587
e|i: 917
t|te: 478
te|t: 271


In [38]:
no_candidate_errors = []

for expected_word, errors in spell_errors.items():
    for misspelled_word in errors:
        candidates = generate_candidates(misspelled_word)

        if not candidates and misspelled_word not in unigrams:
            no_candidate_errors.append(
                (misspelled_word, expected_word)
            )

print("Errors with no valid candidate:", len(no_candidate_errors))
print("Examples:", no_candidate_errors[:20])

Errors with no valid candidate: 17447
Examples: [(' writtings', 'writings'), (' disparingly', 'disparagingly'), ('fore*5', 'four'), ('for*4', 'four'), (' woodes', 'woods'), ('look*2', 'looking'), (' eligble', 'eligible'), (' electrisity', 'electricity'), ('electricty*2', 'electricity'), (' schold', 'scold'), (' adabtable', 'adaptable'), (' imature', 'immature'), (' shoudln', "shouldn't"), (' swival', 'swivel'), (' apropriation', 'appropriation'), (' stabed', 'stabbed'), (' Suothwode', 'Southwold'), (' distrebe', 'disturb'), (' reclections', 'recollections'), (' wensday', 'wednesday')]


In [39]:
import string

no_candidate_reasons = {
    "leading_or_trailing_space": [],
    "special_characters": [],
    "multiple_edits_or_other": []
}

for expected_word, errors in spell_errors.items():
    for misspelled_word in errors:
        candidates = generate_candidates(misspelled_word)

        if candidates or misspelled_word in unigrams:
            continue

        if misspelled_word != misspelled_word.strip():
            no_candidate_reasons["leading_or_trailing_space"].append(
                (misspelled_word, expected_word)
            )
        elif any(ch not in string.ascii_letters for ch in misspelled_word):
            no_candidate_reasons["special_characters"].append(
                (misspelled_word, expected_word)
            )
        else:
            no_candidate_reasons["multiple_edits_or_other"].append(
                (misspelled_word, expected_word)
            )

for reason, examples in no_candidate_reasons.items():
    print(f"\n{reason}: {len(examples)}")
    print(examples[:10])


leading_or_trailing_space: 4938
[(' writtings', 'writings'), (' disparingly', 'disparagingly'), (' woodes', 'woods'), (' eligble', 'eligible'), (' electrisity', 'electricity'), (' schold', 'scold'), (' adabtable', 'adaptable'), (' imature', 'immature'), (' shoudln', "shouldn't"), (' swival', 'swivel')]

special_characters: 1286
[('fore*5', 'four'), ('for*4', 'four'), ('look*2', 'looking'), ('electricty*2', 'electricity'), ('stright*2', 'straight'), ('ingreesh_this', 'increasing'), ('in_cresing', 'increasing'), ('hear*11', 'here'), ('controvercy*2', 'controversy'), ('contraversy*2', 'controversy')]

multiple_edits_or_other: 11223
[('sucesion', 'succession'), ('elligit', 'elegant'), ('incriosing', 'increasing'), ('increcing', 'increasing'), ('ingeker', 'increasing'), ('incrsin', 'increasing'), ('ingcresing', 'increasing'), ('incuiseing', 'increasing'), ('intheris', 'increasing'), ('encreing', 'increasing')]


In [40]:
total = 0
correct_predictions = 0
skipped = 0
wrong_predictions = []
one_edit_errors = 0

for expected_word, errors in spell_errors.items():
    for misspelled_word in errors:

        # Keep only errors at edit distance exactly 1
        edit = identify_edit(expected_word, misspelled_word)

        if edit is None:
            continue

        one_edit_errors += 1

        prediction, scores = correct_word(misspelled_word)

        if not scores:
            skipped += 1
            continue

        total += 1

        if prediction == expected_word:
            correct_predictions += 1
        else:
            wrong_predictions.append(
                (misspelled_word, expected_word, prediction)
            )

incorrect_predictions = total - correct_predictions

accuracy = (
    correct_predictions / total
    if total > 0
    else 0.0
)

coverage = (
    total / one_edit_errors
    if one_edit_errors > 0
    else 0.0
)

print("=== 1-Edit Error Evaluation ===")
print("Total 1-edit errors:", one_edit_errors)
print("Evaluated:", total)
print("Correct predictions:", correct_predictions)
print("Incorrect predictions:", incorrect_predictions)
print("Skipped:", skipped)
print(f"Accuracy on evaluated errors: {accuracy:.2%}")
print(f"Coverage: {coverage:.2%}")

=== 1-Edit Error Evaluation ===
Total 1-edit errors: 8100
Evaluated: 8034
Correct predictions: 4760
Incorrect predictions: 3274
Skipped: 66
Accuracy on evaluated errors: 59.25%
Coverage: 99.19%


In [42]:
total = 0
top1_correct = 0
top3_correct = 0
top5_correct = 0
skipped = 0

for expected_word, errors in spell_errors.items():

    # Only evaluate expected words present in our unigram vocabulary
    if expected_word not in unigrams:
        continue

    for misspelled_word in errors:

        # Keep only 1-edit errors
        if identify_edit(expected_word, misspelled_word) is None:
            continue

        candidates = generate_candidates(misspelled_word)

        # Include the original word if it belongs to the vocabulary
        if misspelled_word in unigrams:
            candidates.append(misspelled_word)

        # Remove duplicate candidates
        candidates = list(set(candidates))

        if not candidates:
            skipped += 1
            continue

        # Rank candidates using our noisy-channel model
        scored_candidates = []

        for candidate in candidates:
            if candidate == misspelled_word:
                score = math.log(word_probability(candidate))
            else:
                score = noisy_channel_score(candidate, misspelled_word)

            scored_candidates.append((candidate, score))

        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        ranked_words = [word for word, score in scored_candidates]

        total += 1

        if ranked_words[0] == expected_word:
            top1_correct += 1

        if expected_word in ranked_words[:3]:
            top3_correct += 1

        if expected_word in ranked_words[:5]:
            top5_correct += 1

# Calculate and display results
print("=== Spell Correction Evaluation ===")
print("Dataset: 1-edit errors, expected words in unigrams")
print("Evaluated:", total)
print("Skipped:", skipped)

if total > 0:
    print(f"Top-1 accuracy: {top1_correct / total:.2%}")
    print(f"Top-3 accuracy: {top3_correct / total:.2%}")
    print(f"Top-5 accuracy: {top5_correct / total:.2%}")
else:
    print("No errors were evaluated.")

=== Spell Correction Evaluation ===
Dataset: 1-edit errors, expected words in unigrams
Evaluated: 7961
Skipped: 0
Top-1 accuracy: 59.79%
Top-3 accuracy: 77.18%
Top-5 accuracy: 82.55%


## Summary of Experiments

We implemented a noisy-channel spell correction model using the unigram frequencies from `count_unigram.txt`, edit-pattern frequencies from `count_1edit.txt`, and spelling errors from `spell-errors.txt`.

### 1. Initial Evaluation

- Evaluated the model on the complete set of spelling errors.
- Obtained **21.49% accuracy** on 22,262 evaluated errors, with 17,447 errors skipped.
- **Reason:** The candidate generator considers only one-edit candidates, so errors requiring multiple edits or containing formatting artifacts often have no valid candidate in the unigram vocabulary.

### 2. Evaluation on One-Edit Errors

- Restricted evaluation to errors differing from the expected word by exactly one insertion, deletion, substitution, or adjacent transposition.
- Obtained **59.25% accuracy** on 8,034 evaluated errors, with 99.19% coverage.
- **Reason:** This experiment evaluates the model on errors that match the assumptions of its one-edit candidate-generation strategy, providing a more appropriate measure of its performance on single-character spelling mistakes.

### 3. Top-1, Top-3, and Top-5 Evaluation

- Further restricted evaluation to one-edit errors whose expected correct words were present in the unigram vocabulary.
- Evaluated 7,961 errors with no skipped cases.

| Metric | Accuracy |
|---|---:|
| Top-1 | 59.79% |
| Top-3 | 77.18% |
| Top-5 | 82.55% |

- **Reason:** Top-1 measures whether the model ranks the correct word first, while Top-3 and Top-5 measure whether it includes the correct word among its three or five highest-ranked candidates. This helps distinguish candidate-generation performance from candidate-ranking performance.

### Conclusion

The model achieves **82.55% Top-5 accuracy** on the filtered one-edit evaluation set. The gap between Top-1 and Top-5 suggests that the correct word is often generated but is not always ranked first. This indicates that improving candidate ranking and edit-channel probability estimation may improve correction accuracy further.

The reported results apply to their respective evaluation subsets and should not be compared directly without accounting for differences in filtering and coverage.